In [ ]:
!pip install --upgrade openai
!pip install -qU langchain-openai

In [28]:
import re
import json
from datetime import datetime
import os

class RMarkdownParser:
    def __init__(self, text):
        self.text = text
        self.sections = {}
        self.r_chunks = []
        self.tables = []
        
    def is_table_separator_line(self, line):
        """Check if a line is a table separator"""
        stripped = line.strip()
        return bool(stripped and set(stripped) <= set('|-: '))
    
    def has_table_structure(self, lines, start_idx):
        """Check if lines starting at start_idx form a valid markdown table"""
        if start_idx + 1 >= len(lines):
            return False
        header = lines[start_idx].strip()
        separator = lines[start_idx + 1].strip()
        if not separator:
            return False
        if not self.is_table_separator_line(separator):
            return False
        return True
    
    def is_main_header(self, line):
        """Check if line is a main header (single #)"""
        stripped = line.strip()
        return stripped.startswith('# ') and not stripped.startswith('## ')
    
    def parse_document(self):
        """Main parsing method with proper table handling"""
        lines = self.text.split('\n')
        
        current_section = "Main"
        current_chunk = []
        in_chunk = False
        current_table = []
        in_table = False
        buffer = []
        
        print("\nDetected headers:")
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # Check for R chunk start
            if line.strip().startswith('```') and ('{r' in line or line.strip() == '```r'):
                in_chunk = True
                i += 1
                continue
                
            # Check for R chunk end    
            if line.strip() == '```':
                if in_chunk:
                    if current_chunk:
                        self.r_chunks.append({
                            'section': current_section,
                            'code': '\n'.join(current_chunk)
                        })
                    current_chunk = []
                    in_chunk = False
                    i += 1
                    continue
            
            # Handle content inside R chunk
            if in_chunk:
                current_chunk.append(line)
                i += 1
                continue
            
            # Check for table start when not in chunk
            if not in_chunk and not in_table:
                if self.has_table_structure(lines, i):
                    in_table = True
                    current_table = [line]
                    i += 1
                    continue
            
            # Handle content inside table
            elif in_table:
                if not line.strip():  # Empty line marks end of table
                    if current_table:
                        self.tables.append({
                            'section': current_section,
                            'content': '\n'.join(current_table)
                        })
                        print(f"Found table in section {current_section}")
                    current_table = []
                    in_table = False
                else:
                    current_table.append(line)
                i += 1
                continue
                
            # Only check for headers if not in chunk and not in table
            if not in_chunk and not in_table and self.is_main_header(line):
                print(f"Found header: {repr(line)}")
                if buffer:
                    if current_section not in self.sections:
                        self.sections[current_section] = []
                    self.sections[current_section].extend(buffer)
                    buffer = []
                current_section = line.strip('# \n')
                i += 1
                continue
            
            # Add to buffer if not in chunk or table
            if not in_chunk and not in_table:
                buffer.append(line)
                
            i += 1
        
        # Handle any remaining content
        if buffer:
            if current_section not in self.sections:
                self.sections[current_section] = []
            self.sections[current_section].extend(buffer)
                
        # Handle any remaining table
        if in_table and current_table:
            self.tables.append({
                'section': current_section,
                'content': '\n'.join(current_table)
            })
    
        print("\nFinal sections found:", list(self.sections.keys()))
        print(f"\nNumber of tables found: {len(self.tables)}")
                
    
    def get_r_libraries(self):
        """Extract all R library calls"""
        libraries = set()
        library_pattern = re.compile(r'(?:library|require)\(["\']?([^)"\']+)["\']?\)')
        
        for chunk in self.r_chunks:
            for line in chunk['code'].split('\n'):
                if 'library(' in line or 'require(' in line:
                    match = library_pattern.search(line)
                    if match:
                        libraries.add(match.group(1))
        return sorted(list(libraries))
    
    def export_json(self, filename="output.json"):
        """Export all content to a JSON file with metadata"""
        output = {
            "metadata": {
                "created_at": datetime.now().isoformat(),
                "total_sections": len(self.sections),
                "total_code_chunks": len(self.r_chunks),
                "total_tables": len(self.tables),
                "r_libraries": self.get_r_libraries()
            },
            "sections": [
                {
                    "title": section,
                    "content": '\n'.join(content).strip(),
                    "word_count": len('\n'.join(content).split())
                }
                for section, content in self.sections.items()
            ],
            "tables": [
                {
                    "section": table["section"],
                    "content": table["content"],
                    "row_count": len(table["content"].split('\n'))
                }
                for table in self.tables
            ],
            "r_code": [
                {
                    "section": chunk["section"],
                    "code": chunk["code"],
                    "line_count": len(chunk["code"].split('\n'))
                }
                for chunk in self.r_chunks
            ]
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        
        return output

def read_rmarkdown(file_path):
    """Read an R Markdown file and return its contents"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except UnicodeDecodeError:
        # Try alternative encodings if utf-8 fails
        try:
            with open(file_path, 'r', encoding='latin-1') as file:
                content = file.read()
            return content
        except Exception as e:
            raise Exception(f"Failed to read file with both UTF-8 and Latin-1 encodings: {str(e)}")
    except Exception as e:
        raise Exception(f"Error reading file: {str(e)}")

def process_rmarkdown_file(file_path, output_dir=None):
    """Process an R Markdown file and export to JSON
    
    Args:
        file_path (str): Path to the input R Markdown file
        output_dir (str, optional): Directory to save the output JSON file. 
                                  If None, saves in the same directory as input file
    
    Returns:
        dict: The parsed content as a dictionary
    """
    # Read the file
    content = read_rmarkdown(file_path)
    
    # Create parser and process content
    parser = RMarkdownParser(content)
    parser.parse_document()
    
    # Get the base filename without extension and path
    base_filename = os.path.splitext(os.path.basename(file_path))[0]
    
    # Generate output path
    if output_dir is None:
        # Use the same directory as input file
        output_dir = os.path.dirname(file_path)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Construct full output path
    output_path = os.path.join(output_dir, f"{base_filename}_parsed.json")
    print(output_path)
    
    # Export to JSON
    output = parser.export_json(output_path)
    
    return output

# Example usage:
"""
# For a single file with custom output directory
output = process_rmarkdown_file("path/to/your/file.Rmd", output_dir="path/to/output")

# For multiple files in a directory
import glob

rmd_files = glob.glob("path/to/directory/*.Rmd")
output_dir = "path/to/output"
for file_path in rmd_files:
    output = process_rmarkdown_file(file_path, output_dir=output_dir)
"""

'\n# For a single file with custom output directory\noutput = process_rmarkdown_file("path/to/your/file.Rmd", output_dir="path/to/output")\n\n# For multiple files in a directory\nimport glob\n\nrmd_files = glob.glob("path/to/directory/*.Rmd")\noutput_dir = "path/to/output"\nfor file_path in rmd_files:\n    output = process_rmarkdown_file(file_path, output_dir=output_dir)\n'

In [34]:
# output = process_rmarkdown_file("rmd_files/08_Unsupervised_Clustering.Rmd", "parsed_docs")
import glob
rmd_files = glob.glob("rmd_files/*.Rmd")
output_dir = "parsed_docs"
for file_path in rmd_files:
    output = process_rmarkdown_file(file_path, output_dir=output_dir)


Detected headers:
Found table in section Main
Found header: '# Motivation'
Found table in section Motivation
Found table in section Motivation
Found table in section Motivation
Found table in section Motivation
Found table in section Motivation
Found table in section Motivation
Found header: '# Foundations of R'
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found table in section Foundations of R
Found header: '# See: http://stats.idre.ucla.edu/r/faq/how-can-i-reshape-my-data-in-r/'
Found header: '# wiki_url <- read_html("https://wiki.socr.umich.edu/index.php/SOCR_Data_PD_BiomedBigMetadata")'
Found header: '# arguments include '
Found header: '# (1) list of variable names that define the different times or metrics (varying), '
Found header: '# (2) the nam

In [23]:
# Test with the provided content
test_content = '''# Hands-on Example: Iris Data

Let's start by seeing a simple example using the Iris dataset, which we saw in [Chapter 2](https://www.socr.umich.edu/people/dinov/2017/Spring/DSPA_HS650/notes/02_ManagingData.html). The data features or attributes include `Sepal.Length`, `Sepal.Width`, `Petal.Length`, and `Petal.Width`, and classes are represented by the `Species taxa` (setosa, versicolor, and virginica).

``` {r eval=T, echo=T, warning=F, message=FALSE}
# install.packages("party")
library("party")
str(iris)
head(iris); table(iris$Species)
```'''

parser = RMarkdownParser(test_content)
parser.parse_document()

print("\nFinal sections:")
print(parser.sections)
print("\nFinal R chunks:")
print(parser.r_chunks)


Starting document parse...

Line 0: '# Hands-on Example: Iris Data'
Current state: in_chunk=False
Found header: '# Hands-on Example: Iris Data'

Line 1: ''
Current state: in_chunk=False

Line 2: "Let's start by seeing a simple example using the Iris dataset, which we saw in [Chapter 2](https://www.socr.umich.edu/people/dinov/2017/Spring/DSPA_HS650/notes/02_ManagingData.html). The data features or attributes include `Sepal.Length`, `Sepal.Width`, `Petal.Length`, and `Petal.Width`, and classes are represented by the `Species taxa` (setosa, versicolor, and virginica)."
Current state: in_chunk=False

Line 3: ''
Current state: in_chunk=False

Line 4: '``` {r eval=T, echo=T, warning=F, message=FALSE}'
Current state: in_chunk=False
Found chunk start marker at line 4

Line 5: '# install.packages("party")'
Current state: in_chunk=True
Adding line to current chunk: '# install.packages("party")'

Line 6: 'library("party")'
Current state: in_chunk=True
Adding line to current chunk: 'library("part